# EDA — Mega Sena Analytics
**Analise Exploratoria Inicial** dos sorteios historicos da Mega Sena (1996–2026).

Dados: `data/sorteios.json` — 3.002 concursos coletados via API da Caixa.
Plots: **Plotly** (interativos — hover, zoom, pan, export PNG pelo menu da figura).

In [49]:
import json
import warnings
from collections import Counter
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

pd.set_option('display.max_columns', None)

warnings.filterwarnings("ignore")

# -- Paths
ROOT      = Path().resolve().parents[1]
DATA_PATH = ROOT / "data" / "sorteios.json"
EXPORTS   = ROOT / "python" / "exports"
EXPORTS.mkdir(exist_ok=True)

# -- Paleta global
PRIMARY   = "#1a1a2e"
ACCENT    = "#e94560"
HIGHLIGHT = "#f5a623"
MUTED     = "#8892b0"
BG        = "#f8f9fa"

# -- Template Plotly customizado
LAYOUT = dict(
    paper_bgcolor=BG,
    plot_bgcolor=BG,
    font=dict(family="DejaVu Sans, sans-serif", color=PRIMARY),
    title_font=dict(size=16, color=PRIMARY),
    legend=dict(bgcolor="rgba(0,0,0,0)", borderwidth=0),
    hoverlabel=dict(bgcolor="white", font_size=12),
    margin=dict(t=70, b=50, l=60, r=30),
)

DEZENA_COLS = ["d1", "d2", "d3", "d4", "d5", "d6"]

print("Imports OK")

Imports OK


## Dados

In [50]:
raw  = json.load(open(DATA_PATH))
rows = []

for c in raw:
    rateo   = {r["faixa"]: r for r in c.get("listaRateioPremio", [])}
    dezenas = sorted([int(x) for x in c["listaDezenas"]])
    sorteio = [int(x) for x in c["dezenasSorteadasOrdemSorteio"]]
    rows.append({
        "numero":       c["numero"],
        "data":         pd.to_datetime(c["dataApuracao"], dayfirst=True),
        "acumulado":    bool(c["acumulado"]),
        "especial":     c["indicadorConcursoEspecial"] != 1,
        "local":        c.get("localSorteio", ""),
        "cidade":       c.get("nomeMunicipioUFSorteio", ""),
        "d1": dezenas[0], "d2": dezenas[1], "d3": dezenas[2],
        "d4": dezenas[3], "d5": dezenas[4], "d6": dezenas[5],
        "s1": sorteio[0], "s2": sorteio[1], "s3": sorteio[2],
        "s4": sorteio[3], "s5": sorteio[4], "s6": sorteio[5],
        "valor_arrecadado":        c.get("valorArrecadado", 0.0),
        "valor_acumulado_proximo": c.get("valorAcumuladoProximoConcurso", 0.0),
        "valor_estimado_proximo":  c.get("valorEstimadoProximoConcurso", 0.0),
        "premio_total_sena":       c.get("valorTotalPremioFaixaUm", 0.0),
        "ganhadores_6": rateo.get(1, {}).get("numeroDeGanhadores", 0),
        "premio_6":     rateo.get(1, {}).get("valorPremio", 0.0),
        "ganhadores_5": rateo.get(2, {}).get("numeroDeGanhadores", 0),
        "premio_5":     rateo.get(2, {}).get("valorPremio", 0.0),
        "ganhadores_4": rateo.get(3, {}).get("numeroDeGanhadores", 0),
        "premio_4":     rateo.get(3, {}).get("valorPremio", 0.0),
    })

df = pd.DataFrame(rows).sort_values("numero").reset_index(drop=True)

def all_dezenas(d=df):
    return d[DEZENA_COLS].values.flatten()

print(f"{len(df)} concursos | {df['data'].min().date()} -> {df['data'].max().date()}")
df.head(3)

3002 concursos | 1996-03-11 -> 2026-05-16


,numero,data,acumulado,especial,local,cidade,d1,d2,d3,d4,d5,d6,s1,s2,s3,s4,s5,s6,valor_arrecadado,valor_acumulado_proximo,valor_estimado_proximo,premio_total_sena,ganhadores_6,premio_6,ganhadores_5,premio_5,ganhadores_4,premio_4
0,1,1996-03-11,True,False,Auditório,"Brasília, DF",4,5,30,33,41,52,41,5,4,52,30,33,0.0,1714650.23,0.0,0.0,0,0.00,17,39158.92,2016,330.21
1,2,1996-03-18,False,False,Caminhão da Sorte,"Belo Horizonte, MG",9,37,39,41,43,49,9,39,37,49,43,41,0.0,0.00,0.0,0.0,1,2307162.23,65,14424.02,4488,208.91
2,3,1996-03-25,False,False,Auditório,"Brasília, DF",10,11,29,30,36,47,36,30,10,11,29,47,0.0,0.00,0.0,0.0,2,391192.51,62,10515.93,4261,153.01


In [51]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3002 entries, 0 to 3001
Data columns (total 28 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   numero                   3002 non-null   int64         
 1   data                     3002 non-null   datetime64[us]
 2   acumulado                3002 non-null   bool          
 3   especial                 3002 non-null   bool          
 4   local                    3002 non-null   str           
 5   cidade                   3002 non-null   str           
 6   d1                       3002 non-null   int64         
 7   d2                       3002 non-null   int64         
 8   d3                       3002 non-null   int64         
 9   d4                       3002 non-null   int64         
 10  d5                       3002 non-null   int64         
 11  d6                       3002 non-null   int64         
 12  s1                       3002 non-null   int6

## 1. Frequencia historica de cada dezena

Quantas vezes cada dezena (1–60) foi sorteada. Barras em destaque = acima da media.

In [52]:
freq   = Counter(all_dezenas())
nums   = list(range(1, 61))
counts = [freq[n] for n in nums]
media  = np.mean(counts)

top5 = sorted(range(1, 61), key=lambda n: freq[n], reverse=True)[:5]
bot5 = sorted(range(1, 61), key=lambda n: freq[n])[:5]

# Cores e opacidades por dezena
colors = []
for n, c in zip(nums, counts):
    if n in top5:
        colors.append(ACCENT)
    elif n in bot5:
        colors.append(HIGHLIGHT)
    elif c >= media:
        colors.append(ACCENT)
    else:
        colors.append(MUTED)

fig = go.Figure(go.Bar(
    x=nums,
    y=counts,
    marker_color=colors,
    text=[f"{c}" for c in counts],
    textposition="outside",
    hovertemplate="Dezena <b>%{x}</b><br>Sorteios: <b>%{y}</b><extra></extra>",
    showlegend=False,
))

fig.add_hline(y=media, line_dash="dash", line_color=PRIMARY, line_width=1.5,
              annotation_text=f"Media: {media:.0f}x",
              annotation_position="top right")

fig.update_layout(
    **LAYOUT,
    title="Frequencia historica de cada dezena (1996-2026)",
    xaxis=dict(title="Dezena", tickmode="linear", tick0=1, dtick=1,
               tickfont=dict(size=9), gridcolor="#e0e0e0"),
    yaxis=dict(title="Num. de sorteios", gridcolor="#e0e0e0"),
    height=420,
)
fig.write_html(EXPORTS / "eda_01_frequencia_dezenas.html")
fig.show()

print(f"Mais frequentes : {sorted(top5)}")
print(f"Menos frequentes: {sorted(bot5)}")

Mais frequentes : [5, 10, 27, 37, 53]
Menos frequentes: [15, 21, 22, 26, 55]


## 2. Frequencia por faixa de dezena

Sorteios agrupados por faixa de 10 numeros. Linha tracejada = distribuicao esperada uniforme.

In [53]:
dezenas   = all_dezenas()
labels    = ["01-10", "11-20", "21-30", "31-40", "41-50", "51-60"]
bins      = [0, 10, 20, 30, 40, 50, 60]
counts, _ = np.histogram(dezenas, bins=bins)
esperado  = dezenas.size / 6
pcts      = [100 * c / dezenas.size for c in counts]

fig = go.Figure(go.Bar(
    x=labels,
    y=counts,
    marker_color=[ACCENT if c > esperado else MUTED for c in counts],
    text=[f"{c:,}<br>({p:.1f}%)" for c, p in zip(counts, pcts)],
    textposition="outside",
    hovertemplate="Faixa <b>%{x}</b><br>Sorteios: <b>%{y:,}</b><br>%{text}<extra></extra>",
))

fig.add_hline(y=esperado, line_dash="dash", line_color=HIGHLIGHT, line_width=2,
              annotation_text=f"Esperado: {esperado:.0f}",
              annotation_position="top right")

fig.update_layout(
    **LAYOUT,
    title="Sorteios por faixa de dezena",
    xaxis=dict(title="Faixa", gridcolor="#e0e0e0", type="category"),
    yaxis=dict(title="Num. de sorteios", gridcolor="#e0e0e0"),
    height=420,
    showlegend=False,
)
fig.write_html(EXPORTS / "eda_02_frequencia_faixa.html")
fig.show()

## 3. Distribuicao de pares e impares por sorteio

Composicao de numeros pares e impares em cada sorteio. A combinacao 3P/3I e a mais comum?

In [54]:
n_pares  = df[DEZENA_COLS].apply(lambda row: (row % 2 == 0).sum(), axis=1)
contagem = n_pares.value_counts().sort_index()
labels   = [f"{p}P / {6-p}I" for p in contagem.index]
pcts     = [100 * v / len(df) for v in contagem.values]

fig = go.Figure(go.Bar(
    x=labels,
    y=contagem.values,
    marker_color=[ACCENT if p == 3 else MUTED for p in contagem.index],
    text=[f"{p:.1f}%" for p in pcts],
    textposition="outside",
    hovertemplate="<b>%{x}</b><br>Sorteios: <b>%{y:,}</b><br>%{text}<extra></extra>",
))

fig.update_layout(
    **LAYOUT,
    title="Distribuicao de pares e impares por sorteio",
    xaxis=dict(title="Composicao par / impar", gridcolor="#e0e0e0", type="category"),
    yaxis=dict(title="Num. de sorteios", gridcolor="#e0e0e0"),
    height=420,
    showlegend=False,
)
fig.write_html(EXPORTS / "eda_03_pares_impares.html")
fig.show()

n_pares.describe()

count    3002.000000
mean        3.016989
std         1.202316
min         0.000000
25%         2.000000
50%         3.000000
75%         4.000000
max         6.000000
dtype: float64

## 4. Distribuicao da soma dos 6 numeros

Pelo Teorema Central do Limite, a soma de 6 variaveis discretas uniformes deve se aproximar de uma normal.

In [55]:
soma = df[DEZENA_COLS].sum(axis=1)

fig = go.Figure()

fig.add_trace(go.Histogram(
    x=soma,
    nbinsx=55,
    marker_color=ACCENT,
    opacity=0.85,
    name="Sorteios",
    hovertemplate="Soma: <b>%{x}</b><br>Freq: <b>%{y}</b><extra></extra>",
))

fig.add_vline(x=soma.mean(), line_dash="dash", line_color=HIGHLIGHT, line_width=2,
              annotation_text=f"Media ({soma.mean():.0f})",
              annotation_position="top right")

fig.add_vline(x=soma.median(), line_dash="dot", line_color=PRIMARY, line_width=2,
              annotation_text=f"Mediana ({soma.median():.0f})",
              annotation_position="bottom right")

fig.update_layout(
    **LAYOUT,
    title="Distribuicao da soma dos 6 numeros por sorteio",
    xaxis=dict(title="Soma dos 6 numeros", gridcolor="#e0e0e0"),
    yaxis=dict(title="Num. de sorteios", gridcolor="#e0e0e0"),
    height=420,
    showlegend=False,
)
fig.write_html(EXPORTS / "eda_04_soma_dezenas.html")
fig.show()

soma.describe()

count    3002.000000
mean      183.160227
std        39.974158
min        66.000000
25%       155.000000
50%       184.000000
75%       211.000000
max       331.000000
dtype: float64

## 5. Gap entre aparicoes de cada dezena

Quantos concursos uma dezena fica sem ser sorteada entre duas aparicoes consecutivas.

In [56]:
gaps_por_dezena = {}
for num in range(1, 61):
    mask      = df[DEZENA_COLS].isin([num]).any(axis=1)
    concursos = df.loc[mask, "numero"].values
    if len(concursos) > 1:
        gaps_por_dezena[num] = np.diff(concursos)

all_gaps   = np.concatenate(list(gaps_por_dezena.values()))
media_gaps = {n: g.mean() for n, g in gaps_por_dezena.items()}

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=["Distribuicao do gap global",
                                    "Gap medio por dezena"])

# Histograma global
fig.add_trace(go.Histogram(
    x=all_gaps, nbinsx=60,
    marker_color=ACCENT, opacity=0.85, name="Gap",
    hovertemplate="Gap: <b>%{x}</b> concursos<br>Freq: <b>%{y}</b><extra></extra>",
), row=1, col=1)
fig.add_vline(x=all_gaps.mean(), line_dash="dash", line_color=HIGHLIGHT,
              line_width=2, row=1, col=1,
              annotation_text=f"Media: {all_gaps.mean():.1f}")

# Gap medio por dezena
nums_g  = list(media_gaps.keys())
meias_g = list(media_gaps.values())
m_ger   = np.mean(meias_g)
fig.add_trace(go.Bar(
    x=nums_g, y=meias_g,
    marker_color=[ACCENT if v > m_ger else MUTED for v in meias_g],
    name="Gap medio",
    hovertemplate="Dezena <b>%{x}</b><br>Gap medio: <b>%{y:.1f}</b> concursos<extra></extra>",
), row=1, col=2)
fig.add_hline(y=m_ger, line_dash="dash", line_color=HIGHLIGHT,
              line_width=2, row=1, col=2,
              annotation_text=f"Media: {m_ger:.1f}")

fig.update_layout(
    **LAYOUT,
    title="Gap entre aparicoes de cada dezena",
    height=430,
    showlegend=False,
)
fig.update_xaxes(gridcolor="#e0e0e0", type="linear")
fig.update_yaxes(gridcolor="#e0e0e0")
fig.write_html(EXPORTS / "eda_05_gap_dezenas.html")
fig.show()

print(f"Gap global: media={all_gaps.mean():.1f} | mediana={np.median(all_gaps):.1f} | max={all_gaps.max()}")

Gap global: media=10.0 | mediana=7.0 | max=109


## 6. Evolucao temporal da frequencia

Frequencia de cada dezena em uma janela deslizante de 200 concursos.
Destaque para as **3 mais** (vermelho) e **3 menos** (amarelo) frequentes no historico total.
As demais dezenas ficam em cinza — clique na legenda para isolar traces.

In [57]:
WINDOW = 200
STEP   = 50

freq_total = Counter(all_dezenas())
top3 = [n for n, _ in freq_total.most_common(3)]
bot3 = [n for n, _ in sorted(freq_total.items(), key=lambda x: x[1])[:3]]

janelas = []
for start in range(0, len(df) - WINDOW + 1, STEP):
    bloco      = df.iloc[start: start + WINDOW]
    freq_bloco = Counter(bloco[DEZENA_COLS].values.flatten())
    c_meio     = int(bloco["numero"].median())
    for d in range(1, 61):
        janelas.append({"concurso": c_meio, "dezena": d, "freq": freq_bloco.get(d, 0)})

df_ev = pd.DataFrame(janelas)

fig = go.Figure()

# Fundo: todas as dezenas em cinza
for dezena in range(1, 61):
    if dezena in top3 or dezena in bot3:
        continue
    sub = df_ev[df_ev["dezena"] == dezena].sort_values("concurso")
    fig.add_trace(go.Scatter(
        x=sub["concurso"], y=sub["freq"],
        mode="lines",
        line=dict(color=MUTED, width=0.5),
        opacity=0.25,
        showlegend=False,
        hoverinfo="skip",
    ))

# Destaque: top3 e bot3
for dezena in top3:
    sub = df_ev[df_ev["dezena"] == dezena].sort_values("concurso")
    fig.add_trace(go.Scatter(
        x=sub["concurso"], y=sub["freq"],
        mode="lines",
        name=f"Dezena {dezena} (top)",
        line=dict(color=ACCENT, width=2.5),
        hovertemplate=f"Dezena {dezena}<br>Concurso: %{{x}}<br>Freq: %{{y}}<extra></extra>",
    ))

for dezena in bot3:
    sub = df_ev[df_ev["dezena"] == dezena].sort_values("concurso")
    fig.add_trace(go.Scatter(
        x=sub["concurso"], y=sub["freq"],
        mode="lines",
        name=f"Dezena {dezena} (bottom)",
        line=dict(color=HIGHLIGHT, width=2.5, dash="dash"),
        hovertemplate=f"Dezena {dezena}<br>Concurso: %{{x}}<br>Freq: %{{y}}<extra></extra>",
    ))

fig.update_layout(
    **LAYOUT,
    title=f"Evolucao temporal da frequencia (janela={WINDOW} concursos, passo={STEP})",
    xaxis=dict(title="Num. do concurso", gridcolor="#e0e0e0"),
    yaxis=dict(title=f"Freq. em janela de {WINDOW} sorteios", gridcolor="#e0e0e0"),
    height=480,
)
fig.write_html(EXPORTS / "eda_06_evolucao_temporal.html")
fig.show()

print(f"Top 3: {top3} | Bottom 3: {bot3}")

Top 3: [np.int64(10), np.int64(53), np.int64(37)] | Bottom 3: [np.int64(26), np.int64(21), np.int64(55)]


## 7. Co-ocorrencia de pares de dezenas

Top 30 pares que mais saem juntos. A linha tracejada marca o valor esperado por distribuicao uniforme:
`C(6,2) / C(60,2) * total_sorteios ≈ 25x`.

In [60]:
pares = []
for row in df[DEZENA_COLS].itertuples(index=False):
    pares.extend(combinations(sorted(row), 2))

freq_pares = Counter(pares)
top30      = freq_pares.most_common(30)
esperado   = len(df) * 15 / 1770

labels = [f"{a}-{b}" for (a, b), _ in top30]
values = [v for _, v in top30]
desvio = [f"{(v - esperado) / esperado * 100:+.1f}%" for v in values]

fig = go.Figure(go.Bar(
    x=values[::-1],
    y=labels[::-1],
    orientation="h",
    marker_color=[ACCENT if v > esperado else MUTED for v in values[::-1]],
    text=[f"{v}x ({d})" for v, d in zip(values[::-1], desvio[::-1])],
    textposition="outside",
    hovertemplate="Par <b>%{y}</b><br>Co-ocorrencias: <b>%{x}</b><br>%{text}<extra></extra>",
))

# Anotação com referência absoluta (yref="paper" = desvinculado do eixo categórico)
fig.add_annotation(
    x=esperado,
    y=0.98,
    yref="paper",
    text=f"<b>Esperado: {esperado:.0f}x</b>",
    showarrow=False,
    bgcolor="rgba(255,255,255,0.9)",
    bordercolor=HIGHLIGHT,
    borderwidth=2,
    borderpad=8,
    font=dict(color=HIGHLIGHT, size=12),
    xanchor="center",
)

layout_custom = dict(**LAYOUT)
layout_custom.update({
    "title": "Top 30 pares de dezenas mais frequentes",
    "xaxis": dict(title="Num. de co-ocorrencias", gridcolor="#e0e0e0"),
    "yaxis": dict(title="Par", gridcolor="#e0e0e0", tickfont=dict(size=10), type="category"),
    "height": 650,
    "showlegend": False,
    "margin": dict(t=70, b=50, l=80, r=120),
})

fig.update_layout(**layout_custom)
fig.write_html(EXPORTS / "eda_07_pares_coocorrencia.html")
fig.show()

print(f"Esperado por par: {esperado:.1f}x")
print(f"Top 5 pares: {top30[:5]}")

Esperado por par: 25.4x
Top 5 pares: [((4, 52), 41), ((5, 27), 41), ((23, 53), 41), ((36, 53), 41), ((38, 53), 40)]
